# W07 -- Data Summary: Target Definition, Feature Engineering, and Modeling

**Two more refinements on top of the previous revision** (which switched the target from `is_unaffordable` to `collapse_onset` and rebuilt the training population around real events):

1. **Confirmed onsets only.** Investigating two leave-one-city-out backtest failures (Springfield, MA and Traverse City, MI, both AUC 0.0) found that their only `collapse_onset` event crossed the 5.0 threshold by a razor-thin margin with zero confirmed quarters afterward -- the label itself was unverifiable. This turned out to generalize: 26 of 168 raw onset events revert below the threshold the very next quarter (most because we can observe the reversion directly, a few because the data window ends right after). A new `collapse_onset_confirmed` label requires the metro to still be unaffordable the following quarter, which rules out both single-quarter statistical noise and unverifiable right-censored events in one rule.
2. **Near-miss cities added as hard negatives.** 33 additional cities reached a price-to-income ratio of 4.5-5.0 at some point but never crossed 5.0. Their data is added to the training population as informative "got close but didn't collapse" negative examples, on top of the 54 cities with a confirmed real event.

**Input:** `data/final_data/price_changes_with_collapse_flags.csv`.

**Outputs:** train/val/holdout CSV splits, metrics tables, and SHAP/risk-ranking figures under `output/`.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import optuna

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedGroupKFold, GroupKFold,
    cross_val_score, cross_val_predict
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report,
    make_scorer, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.discrete.discrete_model import Logit

In [2]:
# Load the R-joined panel (CBSA is the join key across all 5 data sources)
df = pd.read_csv("../../data/final_data/price_changes_with_collapse_flags.csv")

# R exports columns like "metro_name.x" -- flatten dots to underscores for easier access
df.columns = df.columns.str.replace('.', '_', regex=False)

print(df.shape)
print(df.dtypes[['cbsa', 'metro_name_x', 'year', 'qtr']])

(83639, 35)
cbsa            int64
metro_name_x      str
year            int64
qtr             int64
dtype: object


/var/folders/8w/0g4j4t9j4jz3cfq3qwhrsn2r0000gn/T/ipykernel_50196/4133038028.py:2: DtypeWarning: Columns (0: RegionName.y.y) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/final_data/price_changes_with_collapse_flags.csv")


## Target definition: confirmed affordability collapse onset

`collapse_onset` marks the first quarter a metro's price-to-income ratio crosses above **5.0** -- chosen below by comparing onset dates at 4.0 / 4.5 / 5.0 / 5.5 and picking the value that gives the tightest, most realistic cluster of onset dates for Austin, Boise, and Tampa.

`collapse_onset_confirmed` additionally requires the metro to **still be unaffordable the following quarter** -- filtering out both genuine single-quarter reversions (observed directly) and right-censored events at the edge of the data window (where we simply don't have a next quarter to check yet, most recently affected by the 2025 ACS income data lag). This is the target actually used for modeling below.

The modeling population is further restricted to **at-risk rows** (`prev_unaffordable == False`) -- once a metro is already unaffordable, "predicting an onset" for it doesn't mean anything.

In [3]:
# Confirms Austin (12420), Boise (14260), and Tampa (45294) all have data,
# and that the onset dates line up with what we expect: Austin 2021Q2, Boise 2019Q3, Tampa 2021Q4.
ANCHOR_CITIES = {'Austin': 12420.0, 'Boise': 14260.0, 'Tampa': 45294.0}

df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

df['is_unaffordable'] = df['price_to_income_ratio'] > 5.0
df['prev_unaffordable'] = g['is_unaffordable'].shift(1).fillna(False).astype(bool)
df['collapse_onset'] = df['is_unaffordable'] & (~df['prev_unaffordable'])

# Confirmed: still unaffordable the following observed quarter. NaN comparisons
# (a metro with no next quarter yet) evaluate to False, so unverifiable
# right-censored events are excluded the same way as observed reversions.
df['next_qtr_unaffordable'] = g['is_unaffordable'].shift(-1)
df['collapse_onset_confirmed'] = df['collapse_onset'] & (df['next_qtr_unaffordable'] == True)

print("Raw collapse_onset=True rows:", int(df['collapse_onset'].sum()))
print("Confirmed collapse_onset_confirmed=True rows:", int(df['collapse_onset_confirmed'].sum()))
print("Dropped as unconfirmed (single-quarter reversion or right-censored):",
      int(df['collapse_onset'].sum() - df['collapse_onset_confirmed'].sum()))

first_collapse = (
    df[df['collapse_onset_confirmed']]
    .groupby('cbsa')[['metro_name_x', 'year', 'qtr', 'price_to_income_ratio']]
    .first()
)
print("\nConfirmed collapse onset check (anchor cities):")
print(first_collapse.loc[first_collapse.index.isin(ANCHOR_CITIES.values())])

Raw collapse_onset=True rows: 168
Confirmed collapse_onset_confirmed=True rows: 142
Dropped as unconfirmed (single-quarter reversion or right-censored): 26

Confirmed collapse onset check (anchor cities):
                           metro_name_x  year  qtr  price_to_income_ratio
cbsa                                                                     
12420  Austin-Round Rock-San Marcos, TX  2021    2               5.194781
14260                    Boise City, ID  2019    3               5.046829
45294                  Tampa, FL (MSAD)  2021    4               5.213635


In [4]:
# Sensitivity check: does the onset date move much if the threshold isn't 5.0?
for threshold in [4.0, 4.5, 5.0, 5.5]:
    print(f"\n--- Threshold: {threshold} ---")
    for city, code_ in ANCHOR_CITIES.items():
        sub = df[df['cbsa'] == code_].sort_values(['year', 'qtr']).copy()
        sub = sub[sub['price_to_income_ratio'].notna()]
        sub['is_unaffordable'] = sub['price_to_income_ratio'] > threshold
        sub['prev_unaffordable'] = sub['is_unaffordable'].shift(1).fillna(False)
        sub['onset'] = sub['is_unaffordable'] & ~sub['prev_unaffordable']
        onset_row = sub[sub['onset']].head(1)
        if len(onset_row) > 0:
            print(f"  {city}: {onset_row['year'].values[0]}Q{onset_row['qtr'].values[0]}")
        else:
            print(f"  {city}: never crosses this threshold")

# Result: 5.0 gives the tightest, most realistic cluster of onset dates.


--- Threshold: 4.0 ---
  Austin: 2014Q3
  Boise: 2016Q2
  Tampa: 2017Q4

--- Threshold: 4.5 ---
  Austin: 2020Q4
  Boise: 2017Q4
  Tampa: 2021Q2

--- Threshold: 5.0 ---
  Austin: 2021Q2
  Boise: 2019Q3
  Tampa: 2021Q4

--- Threshold: 5.5 ---
  Austin: 2021Q3
  Boise: 2020Q4
  Tampa: 2022Q2


## Feature engineering

Every feature below is lagged by 4 quarters (1 year) before any rolling calculation, so nothing accidentally sees the same-quarter data used to build `collapse_onset`. All 15 features are used by both models.

* Price-to-income level and 5-year change
* ZHVI momentum: YoY, QoQ, and a 3-year rolling trend of YoY
* HPI momentum: YoY and a properly lagged 3-year change
* Population velocity and acceleration
* Rent growth (ZORI YoY)
* Local unemployment rate (level)
* For-sale inventory (QoQ change)
* S&P 500 (YoY change) -- the only feature identical across every metro in a given quarter
* QCEW average weekly wage (YoY change) and employment level (YoY change) -- the income side of the affordability ratio, measured directly

Unemployment, inventory, the S&P 500 return, and both QCEW wage/employment features are left without a monotonic direction assumption; every other feature is assumed to move in the same direction as risk.

**Rule:** never let a feature use current- or future-period information that overlaps with label construction.

In [5]:
LAG = 4

df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

# Price-to-income level and 5-year change
df['price_to_income_lag'] = g['price_to_income_ratio'].shift(LAG)
df['price_to_income_5yr_chg'] = df.groupby('cbsa')['price_to_income_lag'].transform(
    lambda x: x - x.shift(20)
)

# ZHVI momentum -- YoY, QoQ, and a 3yr rolling slope of lagged YoY
df['zhvi_yoy_fixed'] = df.groupby('cbsa')['zhvi_qtr'].transform(lambda x: x.pct_change(4))
df['zhvi_qoq_fixed'] = df.groupby('cbsa')['zhvi_qtr'].transform(lambda x: x.pct_change(1))
df['zhvi_yoy_lag'] = df.groupby('cbsa')['zhvi_yoy_fixed'].shift(LAG)
df['zhvi_qoq_lag'] = df.groupby('cbsa')['zhvi_qoq_fixed'].shift(LAG)
df['three-year_home_price_growth_trend'] = df.groupby('cbsa')['zhvi_yoy_lag'].transform(
    lambda x: x.rolling(12, min_periods=12).apply(
        lambda y: np.polyfit(np.arange(len(y)), y, 1)[0] if y.notna().all() else np.nan
    )
)

# HPI momentum -- YoY (already lagged) and a lagged 3yr change
df['hpi_yoy'] = df.groupby('cbsa')['index_sa'].transform(lambda x: x.pct_change(4))
df['hpi_yoy_lag'] = df.groupby('cbsa')['hpi_yoy'].shift(LAG)
df['hpi_3yr_chg'] = g['index_sa'].transform(lambda x: (x / x.shift(12)) - 1) * 100  # contemporaneous, NOT used as a feature -- reference only
df['hpi_3yr_chg_lag'] = df.groupby('cbsa')['hpi_3yr_chg'].shift(LAG)

# Population velocity and acceleration, lagged
df["pop_yoy_fixed"] = g["population"].transform(lambda x: x.pct_change(4))
df["pop_velocity_fixed"] = df.groupby("cbsa")["pop_yoy_fixed"].diff()
df["pop_velocity_lag"] = df.groupby("cbsa")["pop_velocity_fixed"].shift(LAG)
df["pop_acceleration_fixed"] = df.groupby("cbsa")["pop_velocity_fixed"].diff()
df["pop_acceleration_lag"] = df.groupby("cbsa")["pop_acceleration_fixed"].shift(LAG)

# Rent growth (ZORI YoY), lagged
df["zori_yoy_fixed"] = df.groupby("cbsa")["zori_qtr"].transform(lambda x: x.pct_change(4))
df["zori_yoy_lag"] = df.groupby("cbsa")["zori_yoy_fixed"].shift(LAG)

# Local labor market: unemployment rate level, lagged. Direction left unconstrained.
df["unemployment_rate_lag"] = df.groupby("cbsa")["unemployment_rate"].shift(LAG)

# Housing supply: for-sale inventory, QoQ change. Direction left unconstrained.
df["inv_qoq_fixed"] = df.groupby("cbsa")["inventory_qtr"].transform(lambda x: x.pct_change(1))
df["inv_qoq_lag"] = df.groupby("cbsa")["inv_qoq_fixed"].shift(LAG)

# National S&P 500, YoY change, lagged. Direction left unconstrained.
df["sp500_yoy_fixed"] = df.groupby("cbsa")["sp500_qtr"].transform(lambda x: x.pct_change(4))
df["sp500_yoy_lag"] = df.groupby("cbsa")["sp500_yoy_fixed"].shift(LAG)

# Local wages (QCEW): YoY change in average weekly wage, lagged. Wage growth
# that lags price growth is the mechanism the price-to-income target is built
# on, so this measures the income side of that ratio directly rather than only
# through ACS income (which lags 1-2 years). Direction left unconstrained:
# rising wages can either relieve affordability pressure (higher income) or
# accompany a hot market that outruns them.
df["qcew_wage_yoy_fixed"] = df.groupby("cbsa")["qcew_avg_wkly_wage"].transform(lambda x: x.pct_change(4))
df["qcew_wage_yoy_lag"] = df.groupby("cbsa")["qcew_wage_yoy_fixed"].shift(LAG)

# Local employment level (QCEW): YoY change, lagged. Complements the
# unemployment *rate* -- a metro can hold its rate steady while total jobs
# grow or shrink substantially. Direction left unconstrained.
df["qcew_emp_yoy_fixed"] = df.groupby("cbsa")["qcew_employment"].transform(lambda x: x.pct_change(4))
df["qcew_emp_yoy_lag"] = df.groupby("cbsa")["qcew_emp_yoy_fixed"].shift(LAG)

ALL_FEATURES = [
    "price_to_income_lag",
    "price_to_income_5yr_chg",
    "zhvi_yoy_lag",
    "zhvi_qoq_lag",
    "three-year_home_price_growth_trend",
    "hpi_yoy_lag",
    "hpi_3yr_chg_lag",
    "pop_velocity_lag",
    "pop_acceleration_lag",
    "zori_yoy_lag",
    "unemployment_rate_lag",
    "inv_qoq_lag",
    "sp500_yoy_lag",
    "qcew_wage_yoy_lag",
    "qcew_emp_yoy_lag",
]

## Selecting the training population: confirmed events + near-miss cities

**Event cities:** every city with complete feature data that has at least one *confirmed* `collapse_onset_confirmed` event.

**Near-miss cities (new):** cities that reached a price-to-income ratio of 4.5-5.0 at some point but never crossed 5.0. These contribute only negative examples, but they're a qualitatively different, more informative negative than a permanently-affordable city -- "got close and didn't collapse" is a harder, more useful example for the model to learn from than "was never remotely close."

In [6]:
at_risk = df[~df["prev_unaffordable"]].dropna(subset=ALL_FEATURES + ["collapse_onset_confirmed"]).copy()
event_cities = at_risk.loc[at_risk["collapse_onset_confirmed"], "cbsa"].unique()

print("At-risk rows with complete features:", len(at_risk))
print("Event cities (>=1 confirmed onset, complete features):", len(event_cities))
print("Confirmed collapse_onset_confirmed=True rows:", int(at_risk["collapse_onset_confirmed"].sum()))

# Near-miss cities: reached 4.5-5.0 but never crossed 5.0, anywhere in their history
near_miss_info = df.groupby("cbsa").agg(
    max_pti=("price_to_income_ratio", "max"),
    ever_onset=("collapse_onset", "any"),
).reset_index()
near_miss_cbsas = near_miss_info[
    (near_miss_info["max_pti"] >= 4.5) & (near_miss_info["max_pti"] < 5.0) & (~near_miss_info["ever_onset"])
]["cbsa"]

near_miss_pool = at_risk[at_risk["cbsa"].isin(near_miss_cbsas)]
near_miss_cities = near_miss_pool["cbsa"].unique()
print(f"Near-miss cities (4.5-5.0, never crossed, complete features): {len(near_miss_cities)}")

combined_cities = set(event_cities) | set(near_miss_cities)
training_cbsa_map = (
    at_risk[at_risk["cbsa"].isin(combined_cities)]
    .groupby("cbsa")["metro_name_x"].first()
    .reset_index()
    .set_index("metro_name_x")["cbsa"]
    .to_dict()
)
print(f"\nTotal training cities: {len(training_cbsa_map)} "
      f"({len(event_cities)} event cities + {len(near_miss_cities)} near-miss)")
print("Anchor cities included:", all(c in training_cbsa_map.values() for c in ANCHOR_CITIES.values()))

training_pool_all = at_risk[at_risk["cbsa"].isin(training_cbsa_map.values())].copy()
print(f"\nTraining pool: {len(training_pool_all)} rows, "
      f"positive rate {training_pool_all['collapse_onset_confirmed'].mean():.1%}")

At-risk rows with complete features: 6040
Event cities (>=1 confirmed onset, complete features): 54
Confirmed collapse_onset_confirmed=True rows: 71
Near-miss cities (4.5-5.0, never crossed, complete features): 31

Total training cities: 85 (54 event cities + 31 near-miss)
Anchor cities included: False

Training pool: 1469 rows, positive rate 4.8%


## Findings: why this framing, and what the honest baselines are

- **`is_unaffordable` (the original target) is dominated by persistence** -- it only flips 3.9% of the time over any 4-quarter window, so a trivial "was it already true a year ago" rule beats the fitted model on it.
- **Raw `collapse_onset` includes unverifiable labels** -- 26 of 168 events revert (or can't yet be confirmed) the following quarter.
- **`collapse_onset_confirmed`, restricted to at-risk rows, is the honest target used below.** An "always predict no collapse" baseline scores F1 = 0.0 on it; nothing here can be gamed by persistence or an unconfirmed label.

In [7]:
# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

target = "collapse_onset_confirmed"
model1_pool = training_pool_all.copy()

X_all = model1_pool[ALL_FEATURES]
y_all = model1_pool[target].astype(int)
groups_m1 = model1_pool["cbsa"]

model1_pool.to_csv("output/model1_pool.csv", index=False)
print("Model 1 pool:", len(model1_pool), "rows,", groups_m1.nunique(), "cities,",
      f"{y_all.mean():.1%} positive")

Model 1 pool: 1469 rows, 85 cities, 4.8% positive


In [8]:
# Model 2 uses the same population as Model 1 -- see the Modeling section for
# how the two differ procedurally.
model2_train_cities = dict(training_cbsa_map)
model2_pool = training_pool_all.copy()

Xb_all = model2_pool[ALL_FEATURES]
yb_all = model2_pool[target].astype(int)
groups_m2 = model2_pool["cbsa"]

model2_pool.to_csv("output/model2_pool.csv", index=False)
print("Model 2 pool:", len(model2_pool), "rows,", groups_m2.nunique(), "cities,",
      f"{yb_all.mean():.1%} positive")

Model 2 pool: 1469 rows, 85 cities, 4.8% positive


In [9]:
# Holdout score set: at-risk metros NOT in the training population, complete
# features. These are the metros actually being ranked for early-warning risk.
holdout_scoring = at_risk[~at_risk["cbsa"].isin(training_cbsa_map.values())].dropna(
    subset=ALL_FEATURES
).copy()

print("Holdout scoring rows:", len(holdout_scoring), "| cities:", holdout_scoring["cbsa"].nunique())

holdout_scoring.to_csv("output/holdout_scoring.csv", index=False)
print("Saved: model1_pool.csv, model2_pool.csv, holdout_scoring.csv")

Holdout scoring rows: 4571 | cities: 261


Saved: model1_pool.csv, model2_pool.csv, holdout_scoring.csv


## Modeling: XGBoost + SHAP explainability

With a ~4.7% positive rate (more imbalanced than the event-cities-only population, since near-miss cities add pure-negative rows), both models use `scale_pos_weight`. **PR-AUC (average precision) is the primary reported metric**, always shown next to the no-skill baseline.

In [10]:
os.makedirs("output/figures", exist_ok=True)
os.makedirs("output/tables", exist_ok=True)

ALL_FEATURES = [
    "price_to_income_lag",
    "price_to_income_5yr_chg",
    "zhvi_yoy_lag",
    "zhvi_qoq_lag",
    "three-year_home_price_growth_trend",
    "hpi_yoy_lag",
    "hpi_3yr_chg_lag",
    "pop_velocity_lag",
    "pop_acceleration_lag",
    "zori_yoy_lag",
    "unemployment_rate_lag",
    "inv_qoq_lag",
    "sp500_yoy_lag",
    "qcew_wage_yoy_lag",
    "qcew_emp_yoy_lag",
]
UNCONSTRAINED_FEATURES = {"unemployment_rate_lag", "inv_qoq_lag", "sp500_yoy_lag",
                          "qcew_wage_yoy_lag", "qcew_emp_yoy_lag"}
MONOTONE_INCREASING = tuple(0 if f in UNCONSTRAINED_FEATURES else 1 for f in ALL_FEATURES)

In [11]:
# Reload from the saved splits so this modeling section can be re-run independently
# of the feature-engineering cells above.
model1_pool = pd.read_csv("output/model1_pool.csv")
model2_pool = pd.read_csv("output/model2_pool.csv")
holdout_scoring = pd.read_csv("output/holdout_scoring.csv")

target = "collapse_onset_confirmed"
X_all = model1_pool[ALL_FEATURES]
y_all = model1_pool[target].astype(int)
groups_m1 = model1_pool["cbsa"]

Xb_all = model2_pool[ALL_FEATURES]
yb_all = model2_pool[target].astype(int)
groups_m2 = model2_pool["cbsa"]

### Model 1: Early-Warning Indicator Model

Model 1 identifies which of the engineered indicators is most associated with a *confirmed* `collapse_onset_confirmed` event, across every city with complete feature data that has ever had one, plus the near-miss cities. Default hyperparameters, used for SHAP explainability.

In [12]:
# Grouped split (StratifiedGroupKFold): no city appears on both sides.
neg1, pos1 = (y_all == 0).sum(), (y_all == 1).sum()
scale_pos_weight_1 = neg1 / pos1
print(f"Model 1 class balance: neg={neg1}, pos={pos1}, scale_pos_weight={scale_pos_weight_1:.1f}")

split_m1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx_m1, val_idx_m1 = next(split_m1.split(X_all, y_all, groups=groups_m1))
Xa_train, Xa_val = X_all.iloc[train_idx_m1], X_all.iloc[val_idx_m1]
ya_train, ya_val = y_all.iloc[train_idx_m1], y_all.iloc[val_idx_m1]

train_cities_m1 = set(groups_m1.iloc[train_idx_m1])
val_cities_m1 = set(groups_m1.iloc[val_idx_m1])
print("City overlap between train and val (should be 0):", len(train_cities_m1 & val_cities_m1))
print(f"Train: {len(Xa_train)} rows / {len(train_cities_m1)} cities, "
      f"Val: {len(Xa_val)} rows / {len(val_cities_m1)} cities, "
      f"val positive rate: {ya_val.mean():.1%}")

model1_fresh = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight_1,
    monotone_constraints=MONOTONE_INCREASING,
    random_state=42
)
model1_fresh.fit(Xa_train, ya_train)

pred_fresh = model1_fresh.predict(Xa_val)
proba_fresh = model1_fresh.predict_proba(Xa_val)[:, 1]

baseline_prauc_1 = ya_val.mean()
print("\nFresh Model 1 metrics:")
print({
    "accuracy": accuracy_score(ya_val, pred_fresh),
    "precision": precision_score(ya_val, pred_fresh, zero_division=0),
    "recall": recall_score(ya_val, pred_fresh, zero_division=0),
    "f1": f1_score(ya_val, pred_fresh, zero_division=0),
    "roc_auc": roc_auc_score(ya_val, proba_fresh),
    "pr_auc": average_precision_score(ya_val, proba_fresh),
    "pr_auc_baseline (no-skill)": baseline_prauc_1
})
print(classification_report(ya_val, pred_fresh, zero_division=0))

Model 1 class balance: neg=1398, pos=71, scale_pos_weight=19.7
City overlap between train and val (should be 0): 0
Train: 1210 rows / 69 cities, Val: 259 rows / 16 cities, val positive rate: 5.4%



Fresh Model 1 metrics:


{'accuracy': 0.8803088803088803, 'precision': 0.3023255813953488, 'recall': 0.9285714285714286, 'f1': 0.45614035087719296, 'roc_auc': 0.9282798833819242, 'pr_auc': 0.4696299296213713, 'pr_auc_baseline (no-skill)': np.float64(0.05405405405405406)}
              precision    recall  f1-score   support

           0       1.00      0.88      0.93       245
           1       0.30      0.93      0.46        14

    accuracy                           0.88       259
   macro avg       0.65      0.90      0.69       259
weighted avg       0.96      0.88      0.91       259



In [13]:
model1_metrics = pd.DataFrame([{
    "model": "Model 1 (explanatory, target=collapse_onset_confirmed, event+near-miss population)",
    "n_training_cities": groups_m1.nunique(),
    "accuracy": accuracy_score(ya_val, pred_fresh),
    "precision": precision_score(ya_val, pred_fresh, zero_division=0),
    "recall": recall_score(ya_val, pred_fresh, zero_division=0),
    "f1": f1_score(ya_val, pred_fresh, zero_division=0),
    "roc_auc": roc_auc_score(ya_val, proba_fresh),
    "pr_auc": average_precision_score(ya_val, proba_fresh),
    "pr_auc_baseline": baseline_prauc_1
}])
model1_metrics.to_csv("output/tables/model1_metrics_final.csv", index=False)
print("Saved output/tables/model1_metrics_final.csv")

Saved output/tables/model1_metrics_final.csv


In [14]:
explainer = shap.TreeExplainer(model1_fresh)
shap_values = explainer.shap_values(Xa_train)

plt.figure()
shap.summary_plot(shap_values, Xa_train, show=False)
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model1.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved output/figures/shap_summary_model1.png")

shap_importance = pd.DataFrame({
    "feature": ALL_FEATURES,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance.to_csv("output/tables/shap_importance_model1.csv", index=False)
print(shap_importance)

Saved output/figures/shap_summary_model1.png
                               feature  mean_abs_shap
0                  price_to_income_lag       1.575360
4   three-year_home_price_growth_trend       0.807764
10               unemployment_rate_lag       0.680664
13                   qcew_wage_yoy_lag       0.397917
9                         zori_yoy_lag       0.386594
12                       sp500_yoy_lag       0.271160
3                         zhvi_qoq_lag       0.269811
11                         inv_qoq_lag       0.164243
14                    qcew_emp_yoy_lag       0.151540
2                         zhvi_yoy_lag       0.070524
1              price_to_income_5yr_chg       0.034406
8                 pop_acceleration_lag       0.022638
5                          hpi_yoy_lag       0.012489
7                     pop_velocity_lag       0.007549
6                      hpi_3yr_chg_lag       0.000000


### Model 2: Generalization/Scoring Model

Same training population and full feature set as Model 1 -- confirmed identical by construction, not an accident (see the note after tuning below). Optuna-tuned, including `scale_pos_weight` in the search space; its final fitted version is the one used to score the holdout set.

In [15]:
neg2, pos2 = (yb_all == 0).sum(), (yb_all == 1).sum()
class_ratio_2 = neg2 / pos2
print(f"Model 2 class balance: neg={neg2}, pos={pos2}, class_ratio={class_ratio_2:.1f}")

split_m2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx_m2, val_idx_m2 = next(split_m2.split(Xb_all, yb_all, groups=groups_m2))
Xb_train, Xb_val = Xb_all.iloc[train_idx_m2], Xb_all.iloc[val_idx_m2]
yb_train, yb_val = yb_all.iloc[train_idx_m2], yb_all.iloc[val_idx_m2]
groups_train_m2 = groups_m2.iloc[train_idx_m2]

train_cities_m2 = set(groups_m2.iloc[train_idx_m2])
val_cities_m2 = set(groups_m2.iloc[val_idx_m2])
print("City overlap between train and val (should be 0):", len(train_cities_m2 & val_cities_m2))
print(f"Train: {len(Xb_train)} rows / {len(train_cities_m2)} cities, "
      f"Val: {len(Xb_val)} rows / {len(val_cities_m2)} cities, "
      f"val positive rate: {yb_val.mean():.1%}")

Model 2 class balance: neg=1398, pos=71, class_ratio=19.7
City overlap between train and val (should be 0): 0
Train: 1210 rows / 69 cities, Val: 259 rows / 16 cities, val positive rate: 5.4%


In [16]:
# Hyperparameter + threshold tuning for Model 2 (Optuna, 100 trials, grouped CV on PR-AUC).
Path("output/tables").mkdir(parents=True, exist_ok=True)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)


def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 2.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, class_ratio_2 * 1.5),
        "monotone_constraints": MONOTONE_INCREASING,
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1
    }
    return cross_val_score(
        xgb.XGBClassifier(**params), Xb_train, yb_train,
        groups=groups_train_m2, scoring="average_precision", cv=cv, n_jobs=-1
    ).mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_params = {
    **study.best_params,
    "monotone_constraints": MONOTONE_INCREASING,
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1
}
print("\nBest parameters:", study.best_params)
print(f"Best cross-validation PR-AUC: {study.best_value:.3f} (baseline: {yb_train.mean():.3f})")

base_model = xgb.XGBClassifier(**best_params)
oof_probabilities = cross_val_predict(
    base_model, Xb_train, yb_train, groups=groups_train_m2, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

thresholds = np.arange(0.05, 0.96, 0.01)
f1_scores = [
    f1_score(yb_train, oof_probabilities >= threshold, zero_division=0)
    for threshold in thresholds
]
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"\nOptimal threshold: {best_threshold:.2f}")
print(f"Best OOF F1: {max(f1_scores):.3f}")

model2_final = xgb.XGBClassifier(**best_params)
model2_final.fit(Xb_train, yb_train)

validation_probabilities = model2_final.predict_proba(Xb_val)[:, 1]
validation_predictions = (validation_probabilities >= best_threshold).astype(int)

tuned_metrics = {
    "model": "Model 2 (generalization/scoring, target=collapse_onset_confirmed, event+near-miss population)",
    "n_training_cities": groups_m2.nunique(),
    "features": ", ".join(ALL_FEATURES),
    "threshold": round(best_threshold, 2),
    "accuracy": accuracy_score(yb_val, validation_predictions),
    "precision": precision_score(yb_val, validation_predictions, zero_division=0),
    "recall": recall_score(yb_val, validation_predictions, zero_division=0),
    "f1": f1_score(yb_val, validation_predictions, zero_division=0),
    "roc_auc": roc_auc_score(yb_val, validation_probabilities),
    "pr_auc": average_precision_score(yb_val, validation_probabilities),
    "pr_auc_baseline": yb_val.mean()
}

print("\nTuned Model 2 validation metrics:")
for name, value in tuned_metrics.items():
    print(f"{name}: {value}")
print("\nClassification report:")
print(classification_report(yb_val, validation_predictions, zero_division=0))

[I 2026-08-31 21:01:20,059] A new study created in memory with name: no-name-17928605-21ac-487f-8c9a-117760e1487f


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-08-31 21:01:21,887] Trial 0 finished with value: 0.5039686416987812 and parameters: {'max_depth': 3, 'min_child_weight': 7, 'learning_rate': 0.07259248719561363, 'n_estimators': 140, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.0349658373387986e-08, 'reg_lambda': 0.6245760287469887, 'scale_pos_weight': 18.152943856221707}. Best is trial 0 with value: 0.5039686416987812.


[I 2026-08-31 21:01:23,138] Trial 1 finished with value: 0.42720802437568006 and parameters: {'max_depth': 4, 'min_child_weight': 1, 'learning_rate': 0.13826189316223855, 'n_estimators': 175, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_alpha': 3.3300161336615e-07, 'reg_lambda': 5.472429642032189e-06, 'scale_pos_weight': 15.974035640660759}. Best is trial 0 with value: 0.5039686416987812.
[I 2026-08-31 21:01:23,192] Trial 2 finished with value: 0.4713122694910631 and parameters: {'max_depth': 3, 'min_child_weight': 3, 'learning_rate': 0.05243180891902853, 'n_estimators': 71, 'subsample': 0.7168578594140872, 'colsample_bytree': 0.7465447373174766, 'reg_alpha': 6.107319200689796e-05, 'reg_lambda': 0.11656915613247415, 'scale_pos_weight': 6.69773355849066}. Best is trial 0 with value: 0.5039686416987812.
[I 2026-08-31 21:01:23,255] Trial 3 finished with value: 0.41825412528446587 and parameters: {'max_depth': 3, 'min_child_weight': 5, 'learning_rate': 0.01

[I 2026-08-31 21:01:23,374] Trial 5 finished with value: 0.47084291475307466 and parameters: {'max_depth': 3, 'min_child_weight': 3, 'learning_rate': 0.04089285700048085, 'n_estimators': 132, 'subsample': 0.6739417822102108, 'colsample_bytree': 0.9878338511058234, 'reg_alpha': 0.027189474714697306, 'reg_lambda': 2.8542399074977594, 'scale_pos_weight': 26.53408749248474}. Best is trial 0 with value: 0.5039686416987812.
[I 2026-08-31 21:01:23,425] Trial 6 finished with value: 0.4239100912045871 and parameters: {'max_depth': 3, 'min_child_weight': 7, 'learning_rate': 0.012707942999213693, 'n_estimators': 79, 'subsample': 0.6180909155642152, 'colsample_bytree': 0.7301321323053057, 'reg_alpha': 1.684309275610896e-05, 'reg_lambda': 2.7678419414850017e-06, 'scale_pos_weight': 24.648199909039562}. Best is trial 0 with value: 0.5039686416987812.
[I 2026-08-31 21:01:23,474] Trial 7 finished with value: 0.4632065011551741 and parameters: {'max_depth': 3, 'min_child_weight': 2, 'learning_rate': 0.

...

[I 2026-08-31 21:01:31,233] Trial 88 finished with value: 0.4979295264566316 and parameters: {'max_depth': 3, 'min_child_weight': 2, 'learning_rate': 0.029789538732347725, 'n_estimators': 142, 'subsample': 0.9332359095681833, 'colsample_bytree': 0.8166160545607608, 'reg_alpha': 0.018809762972280743, 'reg_lambda': 0.0711676674969859, 'scale_pos_weight': 6.289502538084983}. Best is trial 59 with value: 0.5274859951440899.
[I 2026-08-31 21:01:31,301] Trial 89 finished with value: 0.4764897812034037 and parameters: {'max_depth': 3, 'min_child_weight': 1, 'learning_rate': 0.026994275010453175, 'n_estimators': 133, 'subsample': 0.9731357298107673, 'colsample_bytree': 0.8314546301971112, 'reg_alpha': 0.04379244403988977, 'reg_lambda': 0.13089760986329513, 'scale_pos_weight': 22.217947584784046}. Best is trial 59 with value: 0.5274859951440899.
[I 2026-08-31 21:01:31,373] Trial 90 finished with value: 0.49999294829365687 and parameters: {'max_depth': 3, 'min_child_weight': 1, 'learning_rate': 

[I 2026-08-31 21:01:31,438] Trial 91 finished with value: 0.5012000106206603 and parameters: {'max_depth': 3, 'min_child_weight': 1, 'learning_rate': 0.05800126627540679, 'n_estimators': 144, 'subsample': 0.9344082150173961, 'colsample_bytree': 0.8409748372283974, 'reg_alpha': 0.005668789481254522, 'reg_lambda': 0.009908330598231293, 'scale_pos_weight': 9.673692881096084}. Best is trial 59 with value: 0.5274859951440899.
[I 2026-08-31 21:01:31,508] Trial 92 finished with value: 0.5031023732141437 and parameters: {'max_depth': 3, 'min_child_weight': 1, 'learning_rate': 0.0648447772637092, 'n_estimators': 138, 'subsample': 0.922291169461576, 'colsample_bytree': 0.841520307286518, 'reg_alpha': 0.003575540989991542, 'reg_lambda': 0.02982104924336154, 'scale_pos_weight': 10.260295312412412}. Best is trial 59 with value: 0.5274859951440899.
[I 2026-08-31 21:01:31,577] Trial 93 finished with value: 0.5011330597848319 and parameters: {'max_depth': 3, 'min_child_weight': 1, 'learning_rate': 0.0

[I 2026-08-31 21:01:31,645] Trial 94 finished with value: 0.48634235659503366 and parameters: {'max_depth': 3, 'min_child_weight': 1, 'learning_rate': 0.07001744383921851, 'n_estimators': 145, 'subsample': 0.8981340843804132, 'colsample_bytree': 0.86533218423989, 'reg_alpha': 0.024853168765042115, 'reg_lambda': 0.04378686869089517, 'scale_pos_weight': 7.4234876279306885}. Best is trial 59 with value: 0.5274859951440899.
[I 2026-08-31 21:01:31,715] Trial 95 finished with value: 0.4968682234158727 and parameters: {'max_depth': 3, 'min_child_weight': 2, 'learning_rate': 0.052747479278359696, 'n_estimators': 140, 'subsample': 0.9125248488907322, 'colsample_bytree': 0.8224489718026824, 'reg_alpha': 0.001258255362141747, 'reg_lambda': 3.5349922551043425, 'scale_pos_weight': 11.295308053548073}. Best is trial 59 with value: 0.5274859951440899.
[I 2026-08-31 21:01:31,788] Trial 96 finished with value: 0.5186288053474566 and parameters: {'max_depth': 3, 'min_child_weight': 1, 'learning_rate': 0

[I 2026-08-31 21:01:31,860] Trial 97 finished with value: 0.5079105666490777 and parameters: {'max_depth': 3, 'min_child_weight': 1, 'learning_rate': 0.034174356069134375, 'n_estimators': 133, 'subsample': 0.9512360638398185, 'colsample_bytree': 0.8044240994585963, 'reg_alpha': 0.0722381576734768, 'reg_lambda': 0.0013942059511299252, 'scale_pos_weight': 16.52680099936514}. Best is trial 59 with value: 0.5274859951440899.
[I 2026-08-31 21:01:31,925] Trial 98 finished with value: 0.5008519907904195 and parameters: {'max_depth': 3, 'min_child_weight': 2, 'learning_rate': 0.03770300515421204, 'n_estimators': 136, 'subsample': 0.989112424531175, 'colsample_bytree': 0.7886451089280007, 'reg_alpha': 0.210491911167567, 'reg_lambda': 0.39910324656946716, 'scale_pos_weight': 17.515866974598413}. Best is trial 59 with value: 0.5274859951440899.
[I 2026-08-31 21:01:31,994] Trial 99 finished with value: 0.49319653880884406 and parameters: {'max_depth': 3, 'min_child_weight': 1, 'learning_rate': 0.0


Optimal threshold: 0.61
Best OOF F1: 0.488

Tuned Model 2 validation metrics:
model: Model 2 (generalization/scoring, target=collapse_onset_confirmed, event+near-miss population)
n_training_cities: 85
features: price_to_income_lag, price_to_income_5yr_chg, zhvi_yoy_lag, zhvi_qoq_lag, three-year_home_price_growth_trend, hpi_yoy_lag, hpi_3yr_chg_lag, pop_velocity_lag, pop_acceleration_lag, zori_yoy_lag, unemployment_rate_lag, inv_qoq_lag, sp500_yoy_lag, qcew_wage_yoy_lag, qcew_emp_yoy_lag
threshold: 0.61
accuracy: 0.918918918918919
precision: 0.3939393939393939
recall: 0.9285714285714286
f1: 0.5531914893617021
roc_auc: 0.9396501457725948
pr_auc: 0.492496244909447
pr_auc_baseline: 0.05405405405405406

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96       245
           1       0.39      0.93      0.55        14

    accuracy                           0.92       259
   macro avg       0.69      0.92      0.75       

In [17]:
model2_metrics = pd.DataFrame([tuned_metrics])
model2_metrics.to_csv("output/tables/model2_metrics_final.csv", index=False)
print("Saved output/tables/model2_metrics_final.csv")

print("\nConfirmed Model 1/Model 2 population identity:",
      model1_pool.sort_values(['cbsa','year','qtr']).reset_index(drop=True).equals(
          model2_pool.sort_values(['cbsa','year','qtr']).reset_index(drop=True)))

Saved output/tables/model2_metrics_final.csv

Confirmed Model 1/Model 2 population identity: True


## City holdout set

Score every at-risk metro outside the training population.

In [18]:
holdout_scoring = pd.read_csv("output/holdout_scoring.csv")
holdout_scoring["risk_score"] = model2_final.predict_proba(holdout_scoring[ALL_FEATURES])[:, 1]

city_risk = (
    holdout_scoring.groupby(["cbsa", "metro_name_x"])["risk_score"]
    .mean().reset_index().sort_values("risk_score", ascending=False)
)

city_risk.to_csv("output/tables/holdout_city_risk_scores.csv", index=False)
print("\nTop 15 highest-risk metros:")
print(city_risk.head(15))


Top 15 highest-risk metros:
      cbsa                        metro_name_x  risk_score
66   20940                       El Centro, CA    0.302884
44   16740   Charlotte-Concord-Gastonia, NC-SC    0.135295
119  27980                 Kahului-Wailuku, HI    0.118634
220  44140                     Springfield, MA    0.115562
257  49420                          Yakima, WA    0.099888
150  32900                          Merced, CA    0.091258
200  41500                         Salinas, CA    0.069383
18   12700                 Barnstable Town, MA    0.065848
260  49700                       Yuba City, CA    0.063919
162  34580          Mount Vernon-Anacortes, WA    0.058248
155  33540                        Missoula, MT    0.055975
157  33700                         Modesto, CA    0.055681
204  41940  San Jose-Sunnyvale-Santa Clara, CA    0.053101
208  42200       Santa Maria-Santa Barbara, CA    0.049977
224  44700                   Stockton-Lodi, CA    0.049807


In [19]:
top15 = city_risk.head(15).sort_values("risk_score", ascending=False)
plt.figure(figsize=(8, 6))
plt.barh(top15['metro_name_x'], top15['risk_score'], color="firebrick")
plt.xlabel("Risk Score")
plt.title("Top 15 Highest-Risk Metros (early-warning: probability of onset)")
plt.tight_layout()
plt.savefig("output/figures/top15_highest_risk_metros.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("Saved output/figures/top15_highest_risk_metros.png")

Saved output/figures/top15_highest_risk_metros.png


/var/folders/8w/0g4j4t9j4jz3cfq3qwhrsn2r0000gn/T/ipykernel_50196/286874796.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Calibrating the holdout risk scores

Percentile rank plus a Platt-scaled probability fit on out-of-fold predictions.

In [20]:
city_risk["risk_percentile"] = city_risk["risk_score"].rank(pct=True)

platt_scaler = LogisticRegression()
platt_scaler.fit(oof_probabilities.reshape(-1, 1), yb_train)

holdout_scoring["risk_score_calibrated"] = platt_scaler.predict_proba(
    holdout_scoring["risk_score"].values.reshape(-1, 1)
)[:, 1]

city_risk_calibrated = (
    holdout_scoring.groupby(["cbsa", "metro_name_x"])["risk_score_calibrated"]
    .mean().reset_index()
)
city_risk = city_risk.merge(city_risk_calibrated, on=["cbsa", "metro_name_x"], how="left")
city_risk.to_csv("output/tables/holdout_city_risk_scores.csv", index=False)

print(city_risk.sort_values("risk_score", ascending=False).head(15))

     cbsa                        metro_name_x  risk_score  risk_percentile  \
0   20940                       El Centro, CA    0.302884         1.000000   
1   16740   Charlotte-Concord-Gastonia, NC-SC    0.135295         0.996169   
2   27980                 Kahului-Wailuku, HI    0.118634         0.992337   
3   44140                     Springfield, MA    0.115562         0.988506   
4   49420                          Yakima, WA    0.099888         0.984674   
5   32900                          Merced, CA    0.091258         0.980843   
6   41500                         Salinas, CA    0.069383         0.977011   
7   12700                 Barnstable Town, MA    0.065848         0.973180   
8   49700                       Yuba City, CA    0.063919         0.969349   
9   34580          Mount Vernon-Anacortes, WA    0.058248         0.965517   
10  33540                        Missoula, MT    0.055975         0.961686   
11  33700                         Modesto, CA    0.055681       

## Validation testing

Grouped cross-validation (`StratifiedGroupKFold`, cities never split across folds) across the full training population.

In [21]:
n_splits_m1 = min(10, groups_m1.nunique())
group_cv = StratifiedGroupKFold(n_splits=n_splits_m1, shuffle=True, random_state=42)

cv_model1 = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    scale_pos_weight=scale_pos_weight_1,
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
auc_scores_m1 = cross_val_score(cv_model1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="roc_auc")
prauc_scores_m1 = cross_val_score(cv_model1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="average_precision")

print(f"Model 1: grouped cross-validation ({n_splits_m1} folds)")
print(f"Mean AUC: {np.nanmean(auc_scores_m1):.3f} (std {np.nanstd(auc_scores_m1):.3f})")
print(f"Mean PR-AUC: {np.nanmean(prauc_scores_m1):.3f} (std {np.nanstd(prauc_scores_m1):.3f}) "
      f"-- baseline (positive rate): {y_all.mean():.3f}")

n_splits_m2 = min(10, groups_m2.nunique())
group_cv_m2 = StratifiedGroupKFold(n_splits=n_splits_m2, shuffle=True, random_state=42)
cv_model2 = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    scale_pos_weight=class_ratio_2,
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
auc_scores_m2 = cross_val_score(cv_model2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="roc_auc")
prauc_scores_m2 = cross_val_score(cv_model2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="average_precision")

print(f"\nModel 2: grouped cross-validation ({n_splits_m2} folds)")
print(f"Mean AUC: {np.nanmean(auc_scores_m2):.3f} (std {np.nanstd(auc_scores_m2):.3f})")
print(f"Mean PR-AUC: {np.nanmean(prauc_scores_m2):.3f} (std {np.nanstd(prauc_scores_m2):.3f}) "
      f"-- baseline (positive rate): {yb_all.mean():.3f}")

cv_results = pd.DataFrame({
    "model": ["model 1"] * len(auc_scores_m1) + ["model 2"] * len(auc_scores_m2),
    "fold": list(range(1, len(auc_scores_m1) + 1)) + list(range(1, len(auc_scores_m2) + 1)),
    "roc_auc": list(auc_scores_m1) + list(auc_scores_m2),
    "pr_auc": list(prauc_scores_m1) + list(prauc_scores_m2),
})
cv_results.to_csv("output/tables/cv_results_final.csv", index=False)
print("\nSaved output/tables/cv_results_final.csv")

Model 1: grouped cross-validation (10 folds)
Mean AUC: 0.940 (std 0.037)
Mean PR-AUC: 0.541 (std 0.174) -- baseline (positive rate): 0.048



Model 2: grouped cross-validation (10 folds)
Mean AUC: 0.940 (std 0.037)
Mean PR-AUC: 0.541 (std 0.174) -- baseline (positive rate): 0.048

Saved output/tables/cv_results_final.csv


## Model comparison: logistic regression baseline

In [22]:
logit_baseline_m1 = make_pipeline(
    StandardScaler(), LogisticRegression(penalty="l2", C=1.0, max_iter=1000, class_weight="balanced")
)
prauc_lr_m1 = cross_val_score(logit_baseline_m1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="average_precision")
auc_lr_m1 = cross_val_score(logit_baseline_m1, X_all, y_all, groups=groups_m1, cv=group_cv, scoring="roc_auc")

logit_baseline_m2 = make_pipeline(
    StandardScaler(), LogisticRegression(penalty="l2", C=1.0, max_iter=1000, class_weight="balanced")
)
prauc_lr_m2 = cross_val_score(logit_baseline_m2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="average_precision")
auc_lr_m2 = cross_val_score(logit_baseline_m2, Xb_all, yb_all, groups=groups_m2, cv=group_cv_m2, scoring="roc_auc")

comparison = pd.DataFrame([
    {"model": "Model 1 XGBoost", "mean_pr_auc": np.nanmean(prauc_scores_m1), "mean_auc": np.nanmean(auc_scores_m1)},
    {"model": "Model 1 Logistic Regression", "mean_pr_auc": np.nanmean(prauc_lr_m1), "mean_auc": np.nanmean(auc_lr_m1)},
    {"model": "Model 2 XGBoost", "mean_pr_auc": np.nanmean(prauc_scores_m2), "mean_auc": np.nanmean(auc_scores_m2)},
    {"model": "Model 2 Logistic Regression", "mean_pr_auc": np.nanmean(prauc_lr_m2), "mean_auc": np.nanmean(auc_lr_m2)},
])
comparison.to_csv("output/tables/model_comparison_logreg_vs_xgboost.csv", index=False)
print(comparison)

                         model  mean_pr_auc  mean_auc
0              Model 1 XGBoost     0.540959  0.940165
1  Model 1 Logistic Regression     0.308519  0.895112
2              Model 2 XGBoost     0.540959  0.940165
3  Model 2 Logistic Regression     0.308519  0.895112


## SHAP explainability (diagnostic: with vs. without the price-to-income level features)

In [23]:
model2_full = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
    scale_pos_weight=class_ratio_2,
    monotone_constraints=MONOTONE_INCREASING, random_state=42
)
model2_full.fit(Xb_all, yb_all)

explainer2 = shap.TreeExplainer(model2_full)
shap_values2 = explainer2.shap_values(Xb_all)

plt.figure()
shap.summary_plot(shap_values2, Xb_all, plot_type="bar", show=False)
plt.title("Model 2 -- all features, target=collapse_onset_confirmed")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model2_full.png", dpi=150)
plt.close()

plt.figure()
shap.summary_plot(shap_values2, Xb_all, show=False)
plt.title("Model 2 -- all features, target=collapse_onset_confirmed")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model2_beeswarm_full.png", dpi=150)
plt.close()

shap_importance_full_m2 = pd.DataFrame({
    "feature": ALL_FEATURES,
    "mean_abs_shap": np.abs(shap_values2).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance_full_m2.to_csv("output/tables/shap_importance_model2_full.csv", index=False)
print("Model 2 (all features) SHAP importance:")
print(shap_importance_full_m2)

Model 2 (all features) SHAP importance:
                               feature  mean_abs_shap
0                  price_to_income_lag       1.488954
4   three-year_home_price_growth_trend       0.828379
10               unemployment_rate_lag       0.663721
13                   qcew_wage_yoy_lag       0.463810
3                         zhvi_qoq_lag       0.381002
9                         zori_yoy_lag       0.318875
12                       sp500_yoy_lag       0.199857
14                    qcew_emp_yoy_lag       0.195728
11                         inv_qoq_lag       0.161572
2                         zhvi_yoy_lag       0.043376
1              price_to_income_5yr_chg       0.033600
8                 pop_acceleration_lag       0.032340
7                     pop_velocity_lag       0.024902
5                          hpi_yoy_lag       0.008336
6                      hpi_3yr_chg_lag       0.008189


In [24]:
NO_LEVEL_FEATURES = [f for f in ALL_FEATURES if f not in ("price_to_income_lag", "price_to_income_5yr_chg")]

cv_model2_noleveL = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
    scale_pos_weight=class_ratio_2, random_state=42
)
auc_noleveL = cross_val_score(
    cv_model2_noleveL, Xb_all[NO_LEVEL_FEATURES], yb_all, groups=groups_m2, cv=group_cv_m2, scoring="roc_auc"
)
prauc_noleveL = cross_val_score(
    cv_model2_noleveL, Xb_all[NO_LEVEL_FEATURES], yb_all, groups=groups_m2, cv=group_cv_m2, scoring="average_precision"
)

diagnostic = pd.DataFrame([
    {"feature_set": "All 15 features", "mean_auc": np.nanmean(auc_scores_m2), "mean_pr_auc": np.nanmean(prauc_scores_m2)},
    {"feature_set": "Without price-to-income level features (13 left)", "mean_auc": np.nanmean(auc_noleveL), "mean_pr_auc": np.nanmean(prauc_noleveL)},
])
diagnostic.to_csv("output/tables/diagnostic_without_level_features.csv", index=False)
print(diagnostic)
print(f"\nBaseline PR-AUC (positive rate): {yb_all.mean():.3f}")

                                        feature_set  mean_auc  mean_pr_auc
0                                   All 15 features  0.940165     0.540959
1  Without price-to-income level features (13 left)  0.863371     0.324503

Baseline PR-AUC (positive rate): 0.048


## Backtesting

**Leave-one-city-out** logistic-regression backtest (statsmodels `Logit`, L1-regularized), reported as a summary distribution given there are 80+ cities now.

In [25]:
def leave_one_city_out_backtest(X_all_bt, y_all_bt, groups, label):
    aucs = {}
    for city_code in sorted(groups.unique()):
        train_mask = groups != city_code
        test_mask = groups == city_code
        y_train_bt, y_test_bt = y_all_bt[train_mask], y_all_bt[test_mask]

        if y_train_bt.nunique() < 2 or y_test_bt.nunique() < 2:
            continue

        X_train_bt = X_all_bt[train_mask].copy()
        X_train_bt.insert(0, "const", 1.0)
        X_test_bt = X_all_bt[test_mask].copy()
        X_test_bt.insert(0, "const", 1.0)

        result = Logit(y_train_bt, X_train_bt).fit_regularized(method="l1", alpha=1.0, disp=0)
        pred_probs = result.predict(X_test_bt)
        auc = roc_auc_score(y_test_bt, pred_probs)
        aucs[city_code] = auc

    if aucs:
        vals = np.array(list(aucs.values()))
        print(f"{label}: {len(aucs)} cities had both classes present in their held-out fold")
        print(f"  mean AUC: {vals.mean():.3f}, median: {np.median(vals):.3f}, "
              f"min: {vals.min():.3f}, max: {vals.max():.3f}")
        worst = sorted(aucs.items(), key=lambda kv: kv[1])[:5]
        best = sorted(aucs.items(), key=lambda kv: -kv[1])[:5]
        city_names = model2_pool.groupby("cbsa")["metro_name_x"].first()
        print("  worst 5:", [(city_names.get(c, c), round(a, 3)) for c, a in worst])
        print("  best 5:", [(city_names.get(c, c), round(a, 3)) for c, a in best])
    return aucs


print("Model 2 backtest")
auc2_by_city = leave_one_city_out_backtest(Xb_all, yb_all, groups_m2, "Model 2")

Model 2 backtest


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


...

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


Model 2: 54 cities had both classes present in their held-out fold
  mean AUC: 0.837, median: 0.917, min: 0.333, max: 1.000
  worst 5: [('Lake Havasu City-Kingman, AZ', 0.333), ('Las Vegas-Henderson-North Las Vegas, NV', 0.333), ('College Station-Bryan, TX', 0.35), ('Boise City, ID', 0.5), ('Burlington-South Burlington, VT', 0.51)]
  best 5: [('Athens-Clarke County, GA', 1.0), ('Austin-Round Rock-San Marcos, TX', 1.0), ('Bridgeport-Stamford-Danbury, CT', 1.0), ('Crestview-Fort Walton Beach-Destin, FL', 1.0), ('Daphne-Fairhope-Foley, AL', 1.0)]
